**E1 - Procesamiento de datos con pandas**

Sesión 14 sep 2026. Dataset: Datos/ventas.csv

In [ ]:
import json
from pathlib import Path

import pandas as pd

**Parte 1 - Diagnóstico**

In [ ]:
DATA_DIR = Path("../Datos")

ventas = pd.read_csv(DATA_DIR / "ventas.csv")
ventas

In [ ]:
print(ventas.shape)
print()
print(ventas.dtypes)
print()
print(ventas.isna().sum())

**Filas que parecen inválidas**

unidades sale como texto (str) en vez de número porque hay una fila con "na" escrito, y hay 2 unidades vacías y 1 precio vacío.

Mirando los datos las filas raras son:

Unidades vacías: C004 (2026-02-01) y C018 (2026-02-11).

Unidades con "na": C022 (2026-02-15).

Unidades a 0 o negativas: C010 con 0 (2026-02-05), C012 con -2 (2026-02-07) y C030 con 0 (2026-02-19).

Precio vacío: C025 (2026-02-17).

Precio a 0 o negativo: C015 con 0 (2026-02-09), C030 con -1 (2026-02-19), C007 con -5 (2026-02-03) y C020 con -32 (2026-02-13).

En total son 10 filas (la de C030 tiene mal las dos cosas).

**Parte 2 - Validación**

In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = frame.copy()
    df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")

    ok = (df["unidades"] > 0) & (df["precio_unitario"] > 0)

    validos = df[ok].copy()
    errores = df[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(ventas)
print("válidas:", len(validos), "| inválidas:", len(errores))
errores

Me salen 140 válidas y 10 inválidas, que coincide con el checkpoint del enunciado y con las filas que había visto en el diagnóstico.

Con errors="coerce" el "na" se convierte en NaN, y no hace falta comprobar los nulos aparte porque NaN > 0 da False.

**Parte 3 - Agregaciones**

In [ ]:
# importe total por region
por_region = validos.groupby("region")["importe"].sum().sort_values(ascending=False)
por_region

In [ ]:
# top 3 productos
top_productos = validos.groupby("producto")["importe"].sum().sort_values(ascending=False).head(3)
top_productos

In [ ]:
# clientes con mas de una compra
compras = validos["cliente_id"].value_counts()
clientes_repetidos = compras[compras > 1]
clientes_repetidos

**Parte 4 - Exportación**

In [ ]:
validos.to_csv(DATA_DIR / "ventas_limpias.csv", index=False)

calidad = {
    "filas_totales": len(ventas),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": round(float(validos["importe"].sum()), 2),
}

with open(DATA_DIR / "calidad_datos.json", "w", encoding="utf-8") as f:
    json.dump(calidad, f, indent=2)

calidad

**Extensión - ventas por semana**

In [ ]:
validos["fecha"] = pd.to_datetime(validos["fecha"])
validos["semana"] = validos["fecha"].dt.isocalendar().week
validos.groupby("semana")["importe"].sum()